# Adversarial Training as a Mitigation for White-Box FGSM Evasion Attacks

**Framework:** PyTorch  
**Dataset:** MNIST   
**Duration:** 3 hours

### Purpose

This laboratory studies **adversarial training** as a defense against white-box FGSM evasion attacks. The mechanics of FGSM are assumed knowledge. The focus is the design of the mitigation: how adversarial training data should be generated, how much adversarial data should be used, how the training objective should be defined, how the main hyperparameters should be selected, and how the resulting defense should be evaluated.

The laboratory considers both **untargeted** and **targeted** FGSM attacks under an $$\ell_\infty$$ perturbation constraint.

### Learning objectives

By the end of the laboratory, you should be able to:

- design a controlled adversarial-training experiment;
- compare static adversarial augmentation with online adversarial training;
- study the effect of adversarial coverage, loss weighting, and perturbation budget;
- implement untargeted and targeted adversarial training;
- evaluate a defended model using fresh white-box attacks;
- select a defense using validation evidence;
- distinguish an empirical robustness result from a general robustness guarantee;
- identify unnecessary computation in an adversarial-training implementation.

### Programming requirement

You must write all executable code. Each code cell contains only comments describing the required task. You are responsible for the implementation.


## Background: adversarial training

A clean classifier is trained by updating the model parameters $\theta$ so that the prediction loss is small on clean training examples.

For a mini-batch of size $B$, define the clean mini-batch cost as

$$
J_{\mathrm{clean}}(\theta)
=
\frac{1}{B}
\sum_{i=1}^{B}
\mathcal{L}\left(f_\theta(x_i),y_i\right).
$$

Adversarial training changes the **training procedure**. During training, some clean inputs are perturbed by an attack, and the resulting adversarial inputs are used to update the model. The ground-truth class remains the correct training label.

Let $m$ be the number of examples selected for adversarial generation in a mini-batch of size $B$. Define the **adversarial coverage**

$$
\rho = \frac{m}{B},
\qquad
0 < \rho \leq 1.
$$

The perturbation budget used while training the defended model is denoted by $\epsilon_{\mathrm{train}}$.

The attack budget used when evaluating the trained model is denoted by $
\epsilon_{\mathrm{test}}$.

These quantities have different roles. The first is a training design choice; the second is part of the evaluation condition.

For the selected adversarial examples, define $J_{\mathrm{adv}}(\theta)$
as the average classification cost on the adversarial inputs, using their correct labels. 

A weighting parameter $\lambda \in [0,1]$ will be used later to control the relative contribution of clean and adversarial costs.

Two attack objectives will be considered during training:

- **untargeted adversarial training:** adversarial examples are generated to cause any incorrect prediction;
- **targeted adversarial training:** adversarial examples are generated toward a specified target class, but the defender still trains them with their correct labels.

To isolate the effect of each design choice, Sections 1–5 first use **untargeted FGSM** for adversarial training. Section 6 then compares untargeted and targeted adversarial training.

### Evaluation quantities

You will report at least:

- **clean accuracy:** accuracy on unmodified inputs;
- **robust accuracy:** accuracy after the specified adversarial attack;
- **attack success rate (ASR):** the fraction of eligible attacks that satisfy the attack objective.

All model choices must be made using training and validation data. The held-out test set is reserved for the final evaluation.


## 1. Establish a controlled baseline

A defense can only be evaluated fairly if the defended and undefended models are compared under controlled conditions. Before implementing adversarial training, define the experimental protocol that will remain fixed throughout the laboratory.


### Question 1 — What must be controlled?

1. Which model, data, and optimization settings should remain the same when comparing clean training with adversarial training?
2. Why is a clean baseline necessary?
3. During model development, what is the role of the validation set?
4. What is the role of the test set?
5. Which quantities should be recorded so that the experiment can be reproduced?

State your experimental protocol before writing code.


**Write your response here:**

1. we need to keep : 
     * The architechture
     * The Dataset
     * Train/Test/Validation proportions
     * The optimizer, learning rate, the batch size, the loss
     * All the preprocessing and normalization <br><br>
2. The baseline is use as a control, it show what the same model and trainning pipeline can achive with and without adverserial exemples<br><br>
3. During the deployment the validation set is use to choose the rights hyperparameter <br><br>
4. The tes set is use at the end for the final evaluation of the model, we use the same set for the baseline and for the adversarial trainning to reveal the performances of each approah<br><br>
5. We need to keep the same the dataset and the train/test propotion and détail and the trainning and evaluation settings to get the comparaison<br><br>

### Implementation 1 — Experimental setup and clean baseline

Implement the complete experimental setup. Use one fixed architecture and one fixed data protocol for all later comparisons.


In [47]:
# 1. Import the libraries required for data handling, modelling, timing, tables, and plots.
import random
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, transforms


In [18]:

# 2. Set the random seeds needed for reproducibility and select the available compute device.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Device:", device)


PyTorch version: 2.14.1+cpu
Device: cpu


In [19]:
# 3. Load MNIST with pixel values represented on the [0, 1] scale.
transform = transforms.ToTensor()

# 4. Create disjoint training and validation sets from the official MNIST training split.
#    Keep the official test set untouched.
full_train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
train_dataset, validation_dataset = random_split(
    full_train_dataset,
    [50_000, 10_000],
    generator=torch.Generator().manual_seed(SEED),
)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)


In [20]:
# 5. Create the DataLoader objects required for training, validation, and testing.
#    You may also create one fixed smaller study subset for the parameter-sweep experiments.
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False)

print("Number of training samples:", len(train_dataset))
print("Number of test samples:", len(test_dataset))
print("Number of validation samples:", len(validation_loader.dataset))


Number of training samples: 60000
Number of test samples: 10000
Number of validation samples: 10000


In [21]:
# 6. Implement the fixed CNN architecture that will be used in every comparison.
class SmallCNN(nn.Module):
    def __init__(self):
        super(SmallCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = x.view(-1, 64 * 7 * 7)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SmallCNN().to(device)
print(model)



SmallCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [ ]:
# 7. Implement clean training.
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 3

def train_model():
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total if total > 0 else 0
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")



In [23]:
# 8. Implement clean-accuracy evaluation.
def evaluate_model():
     model.eval()
     correct = 0
     total = 0

     with torch.no_grad():
          for x_batch, y_batch in test_loader:
               x_batch = x_batch.to(device)
               y_batch = y_batch.to(device)

               logits = model(x_batch)
               predictions = logits.argmax(dim=1)

               correct += (predictions == y_batch).sum().item()
               total += y_batch.size(0)

     clean_accuracy = correct / total
     print(f"Clean test accuracy: {100 * clean_accuracy:.2f}%")

In [24]:
# 9. Train the clean baseline and record the settings, training time, and validation accuracy.
train_model()
evaluate_model()

Epoch [1/3], Loss: 0.1884, Accuracy: 0.9424
Epoch [2/3], Loss: 0.0503, Accuracy: 0.9849
Epoch [3/3], Loss: 0.0339, Accuracy: 0.9896
Clean test accuracy: 98.97%


## 2. Choose when adversarial examples are generated

There are two natural ways to introduce adversarial inputs into training.

**Static adversarial augmentation:** adversarial examples are generated once using a fixed model and then reused.

**Online adversarial training:** adversarial examples are generated while the defended model is being trained.

The two procedures use adversarial data differently. Their effect should be established experimentally rather than assumed.


### Question 2 — Static or online generation?

Suppose the static adversarial set is generated using model parameters $\theta_0$. The defended model is then updated through $\theta_1,\theta_2,\ldots,\theta_T$.

1. Which parameters were used to construct the static adversarial examples?
2. What changes after each update from $\theta_t$ to $\theta_{t+1}$?
3. How might these changes affect the usefulness of a fixed adversarial set?
4. Compare the storage cost and computation cost of static and online generation.
5. How would you make an experimental comparison between the two procedures fair?
6. State your hypothesis before running the experiment.


**Write your response here:**

1. The static adversarial examples were generated using the model’s initial parameters, $\theta_0$. They are then held fixed while the defended model trains<br><br>
2. The model parameters change from $\theta_t$ to $\theta_{t+1}$, at the same time the loss and the accuracy can also change<br><br>
3. A fixed example that was adversarial for the initial model may become less challenging or no longer adversarial as training changes the model.<br><br>
4. <br><br>
5. Keep the architecture, initialization, data splits and preprocessing, optimizer and schedule, batch size, number of training updates, and clean/adversarial example proportions the same<br><br>
6. <br><br>


### Implementation 2 — Compare static and online generation

Implement both procedures using untargeted FGSM. Use the same clean data, architecture, optimizer settings, perturbation budget, and number of training epochs.


In [49]:
# 1. Implement a reusable untargeted FGSM generation function for a mini-batch.
#    The function should return detached adversarial inputs in the valid input range.
def generate_fgsm_adversarial_inputs(model, x_batch, y_batch, epsilon):
    if epsilon < 0:
        raise ValueError("epsilon must be non-negative")

    inputs_for_attack = x_batch.detach().clone().requires_grad_(True)
    with torch.enable_grad():
        logits = model(inputs_for_attack)
        loss = F.cross_entropy(logits, y_batch)
        input_gradient, = torch.autograd.grad(loss, inputs_for_attack)

    with torch.no_grad():
        adversarial_inputs = inputs_for_attack + epsilon * input_gradient.sign()
        adversarial_inputs = torch.clamp(adversarial_inputs, 0, 1)

    return adversarial_inputs.detach()


In [50]:
# 2. Implement static adversarial augmentation:
#    generate the required adversarial training examples once and store them.
def generate_static_adversarial_dataset(model, data_loader, epsilon):
    was_training = model.training
    model.eval()
    adversarial_batches = []
    label_batches = []

    try:
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)
            adversarial_images = generate_fgsm_adversarial_inputs(
                model, images, labels, epsilon
            )
            adversarial_batches.append(adversarial_images.cpu())
            label_batches.append(labels.cpu())
    finally:
        model.train(was_training)

    if not adversarial_batches:
        raise ValueError("Cannot generate adversarial examples from an empty loader")

    return TensorDataset(torch.cat(adversarial_batches), torch.cat(label_batches))


# Freeze the clean baseline parameters as theta_0 before constructing the static set.
theta_0_state = {
    name: parameter.detach().clone()
    for name, parameter in model.state_dict().items()
}
static_attack_model = SmallCNN().to(device)
static_attack_model.load_state_dict(theta_0_state)

epsilon_train = 0.15
static_generation_loader = DataLoader(
    train_dataset, batch_size=train_loader.batch_size, shuffle=False
)
generation_start = time.perf_counter()
static_adv_dataset = generate_static_adversarial_dataset(
    static_attack_model, static_generation_loader, epsilon_train
)
static_generation_seconds = time.perf_counter() - generation_start

print(f"Stored adversarial examples: {len(static_adv_dataset)}")
print(f"Static generation time: {static_generation_seconds:.2f} seconds")


Stored adversarial examples: 60000
Static generation time: 86.38 seconds


In [52]:
# 3. Train a fresh model using the static adversarial data and the corresponding clean data.
class StaticAdversarialPairDataset(torch.utils.data.Dataset):
    def __init__(self, clean_dataset, adversarial_dataset):
        if len(clean_dataset) != len(adversarial_dataset):
            raise ValueError("Clean and static adversarial datasets must have equal lengths")
        self.clean_dataset = clean_dataset
        self.adversarial_dataset = adversarial_dataset

    def __len__(self):
        return len(self.clean_dataset)

    def __getitem__(self, index):
        clean_image, clean_label = self.clean_dataset[index]
        adversarial_image, adversarial_label = self.adversarial_dataset[index]
        if not torch.equal(torch.as_tensor(clean_label), torch.as_tensor(adversarial_label)):
            raise ValueError(f"Label mismatch for training example {index}")
        return clean_image, clean_label, adversarial_image, adversarial_label


def train_model_with_static_adversarial_data(
    model, clean_dataset, adversarial_dataset, batch_size, epochs, learning_rate
):
    paired_dataset = StaticAdversarialPairDataset(clean_dataset, adversarial_dataset)
    paired_loader = DataLoader(paired_dataset, batch_size=batch_size, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    start_time = time.perf_counter()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for clean_images, clean_labels, adversarial_images, adversarial_labels in paired_loader:
            clean_images = clean_images.to(device)
            clean_labels = clean_labels.to(device)
            adversarial_images = adversarial_images.to(device)
            adversarial_labels = adversarial_labels.to(device)

            if not torch.equal(clean_labels, adversarial_labels):
                raise ValueError("Clean and adversarial batch labels do not match")

            inputs = torch.cat((clean_images, adversarial_images), dim=0)
            labels = torch.cat((clean_labels, adversarial_labels), dim=0)

            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_accuracy = correct / total
        history.append({"loss": epoch_loss, "accuracy": epoch_accuracy})
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}"
        )

    return history, time.perf_counter() - start_time


# Start from a fresh initialization; each update sees paired clean and static adversarial inputs.
static_model = SmallCNN().to(device)
static_initial_state = {
    name: tensor.detach().clone()
    for name, tensor in static_model.state_dict().items()
}
static_training_history, static_training_seconds = train_model_with_static_adversarial_data(
    static_model,
    train_dataset,
    static_adv_dataset,
    batch_size=train_loader.batch_size,
    epochs=epochs,
    learning_rate=0.001,
)
print(f"Static adversarial training time: {static_training_seconds:.2f} seconds")


Epoch [1/3], Loss: 0.2221, Accuracy: 0.9314
Epoch [2/3], Loss: 0.0456, Accuracy: 0.9864
Epoch [3/3], Loss: 0.0284, Accuracy: 0.9910
Static adversarial training time: 553.76 seconds


In [53]:
# 4. Implement online adversarial training:
#    generate adversarial examples inside the training loop while the model parameters are changing.
def train_model_online_adversarial(model, data_loader, epsilon, epochs, learning_rate):
    if epsilon < 0:
        raise ValueError("epsilon must be non-negative")
    if epochs <= 0:
        raise ValueError("epochs must be positive")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    adversarial_examples_generated = 0
    start_time = time.perf_counter()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for clean_images, clean_labels in data_loader:
            clean_images = clean_images.to(device)
            clean_labels = clean_labels.to(device)

            # Generate each batch's perturbations from the model's current parameters.
            adversarial_images = generate_fgsm_adversarial_inputs(
                model, clean_images, clean_labels, epsilon
            )
            adversarial_examples_generated += clean_labels.size(0)

            inputs = torch.cat((clean_images, adversarial_images), dim=0)
            labels = torch.cat((clean_labels, clean_labels), dim=0)

            optimizer.zero_grad()
            logits = model(inputs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)

        epoch_loss = running_loss / total
        epoch_accuracy = correct / total
        history.append({"loss": epoch_loss, "accuracy": epoch_accuracy})
        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.4f}"
        )

    return history, time.perf_counter() - start_time, adversarial_examples_generated


# Match the static procedure's initialization, optimizer, batch size, and update budget.
online_model = SmallCNN().to(device)
online_model.load_state_dict(static_initial_state)
online_training_history, online_training_seconds, online_adversarial_examples_generated = (
    train_model_online_adversarial(
        online_model,
        train_loader,
        epsilon=epsilon_train,
        epochs=epochs,
        learning_rate=0.001,
    )
)
print(f"Online adversarial training time: {online_training_seconds:.2f} seconds")
print(
    "Online adversarial examples generated: "
    f"{online_adversarial_examples_generated}"
)


Epoch [1/3], Loss: 0.3557, Accuracy: 0.8853
Epoch [2/3], Loss: 0.1446, Accuracy: 0.9527
Epoch [3/3], Loss: 0.0988, Accuracy: 0.9677
Online adversarial training time: 670.79 seconds
Online adversarial examples generated: 180000


In [ ]:
# 5. Evaluate both defended models on the same validation set using fresh white-box attacks.


In [59]:
# 6. Record clean accuracy, robust accuracy, training time, and the number of adversarial examples generated.
print(f"Stored adversarial examples: {len(static_adv_dataset)}")
print(f"Static generation time: {static_generation_seconds:.2f} seconds")
print(f"Training perturbation budget: {epsilon_train}")



Stored adversarial examples: 60000
Static generation time: 86.38 seconds
Training perturbation budget: 0.15


In [ ]:
# 7. Present the comparison in one table.


### Question 3 — What did the comparison show?

1. Did the result support your hypothesis?
2. Did one procedure improve robustness at the cost of more computation or storage?
3. If the two procedures produced similar results, what additional experiment would you run before concluding that they are equivalent?
4. Based on your evidence, which procedure will you use in the remaining sections? Justify the choice.


**Write your response here:**

<br><br><br><br>


## 3. Choose adversarial coverage

Online adversarial training does not require every clean example in a mini-batch to be perturbed.

For a mini-batch of size $B$, if $m$ examples are selected for attack, $\rho=\frac{m}{B}$.

The value of $\rho$ controls the amount of adversarial exposure during training.


### Question 4 — How much adversarial data is needed?

1. For a mini-batch of size $B=256$, how many examples are attacked when $\rho=0.25$, $0.50$, and $1.00$?
2. Distinguish the **stored dataset size** from the **number of adversarial examples generated during training**.
3. Should the same examples be selected for attack in every epoch, or should the selection change? What are the consequences of each choice?
4. Compare random selection with selecting examples that currently have a high loss. What bias could each strategy introduce?
5. What measurements are needed to decide whether full coverage is worth the extra computation?
6. State your expected trend as $\rho$ increases.


**Write your response here:**

<br><br><br><br><br>


### Implementation 3 — Study the effect of $\rho$

Extend the online training procedure so that only a selected fraction of each mini-batch is used for adversarial generation.


In [26]:
# 1. Add a coverage parameter rho to the online adversarial-training procedure.
#
# 2. Implement and document the rule used to select the examples that will be attacked.
#
# 3. Count how many adversarial examples are generated during training.
#
# 4. Train comparable models for at least three rho values, including partial and full coverage.
#
# 5. Keep the remaining settings fixed.
#
# 6. Measure clean validation accuracy, robust validation accuracy, and training time.
#
# 7. Plot or tabulate the results as a function of rho.


## 4. Define the adversarial-training objective

Adversarial training must specify how clean and adversarial performance contribute to the parameter update.

For a mini-batch, let $J_{\mathrm{clean}}(\theta)$ denote the clean cost and $J_{\mathrm{adv}}(\theta)$ denote the cost on the selected adversarial examples.

A common weighted objective is

$$
J_{\mathrm{AT}}(\theta)
=
\lambda J_{\mathrm{clean}}(\theta)
+
(1-\lambda)J_{\mathrm{adv}}(\theta),
$$

where $0\leq\lambda\leq1$.

The value of $\lambda$ is a design choice; it is not determined by FGSM.


### Question 5 — What does $\lambda$ control?

1. What training procedure is obtained when $\lambda=1$?
2. What training procedure is obtained when $\lambda=0$?
3. Why is $\lambda=0.5$ not automatically the best choice?
4. If $J_{\mathrm{adv}}$ is much larger than $J_{\mathrm{clean}}$, does $\lambda=0.5$ necessarily mean that the two terms have the same influence on the update?
5. What clean-accuracy and robustness trade-off do you expect as $\lambda$ changes?
6. Define a validation criterion that could be used to choose $\lambda$.


**Write your response here:**

<br><br><br><br><br>


### Implementation 4 — Study $\lambda$

Use the online adversarial-training procedure selected in Section 2. Keep $\rho$ and $\epsilon_{\mathrm{train}}$ fixed while varying $\lambda$.


In [27]:
# 1. Modify the online training loop so that it optimizes J_AT with a configurable lambda.
#
# 2. Choose a small grid of lambda values that includes both extreme and intermediate cases.
#
# 3. Train comparable models for all selected lambda values.
#
# 4. Evaluate clean and robust validation accuracy for each model.
#
# 5. Record the training time for each setting.
#
# 6. Apply the validation criterion stated in Question 5.
#
# 7. Present the clean/robustness trade-off in a table or plot.


### Question 6 — Can $\lambda$ be updated automatically?

Suppose $\lambda$ itself is treated as a trainable parameter in

$$
J_{\mathrm{AT}}
=
\lambda J_{\mathrm{clean}}
+
(1-\lambda)J_{\mathrm{adv}}.
$$

1. Derive $\frac{\partial J_{\mathrm{AT}}}{\partial\lambda}$.

2. If $J_{\mathrm{adv}} > J_{\mathrm{clean}}$, what direction would gradient descent move $\lambda$?
3. Would this necessarily select the best robustness/clean-accuracy trade-off?
4. Propose a better rule for selecting or adapting $\lambda$.
5. What information would your rule need that ordinary parameter training does not use?


**Write your response here:**

<br><br><br><br><br>


## 5. Choose the training perturbation budget

The perturbation budget used during adversarial training is $\epsilon_{\mathrm{train}}$.

The perturbation budget used to evaluate a trained model is $\epsilon_{\mathrm{test}}$.

A model trained at one perturbation budget should not be assumed to behave the same way at other budgets.


### Question 7 — How strong should the training attack be?

1. What risk do you expect if $\epsilon_{\mathrm{train}}$ is very small?
2. What risk do you expect if $\epsilon_{\mathrm{train}}$ is very large?
3. Why should the same defended model be evaluated over several values of $\epsilon_{\mathrm{test}}$?
4. What would you expect when $\epsilon_{\mathrm{test}} > \epsilon_{\mathrm{train}}$?
5. How would you choose $\epsilon_{\mathrm{train}}$ from validation evidence?
6. What additional concern arises if the perturbation becomes so large that the semantic label may no longer be clear?


**Write your response here:**

<br><br><br><br><br>


### Implementation 5 — Build a training/evaluation budget matrix

Train several models with different values of $\epsilon_{\mathrm{train}}$ and evaluate every model using the same grid of $\epsilon_{\mathrm{test}}$ values.


In [28]:
# 1. Choose a small grid of epsilon_train values on the [0, 1] input scale.
#
# 2. Train one comparable defended model for each epsilon_train value.
#
# 3. Choose a common epsilon_test grid that includes values below, equal to, and above the training budgets.
#
# 4. Implement white-box validation evaluation for every trained model and every epsilon_test value.
#
# 5. Avoid recomputing quantities that are mathematically identical within one FGSM epsilon sweep.
#
# 6. Record clean accuracy and robust accuracy.
#
# 7. Present the robust-accuracy results as a matrix or heat map.


### Question 8 — What does the budget matrix tell you?

1. Does the largest value of $\epsilon_{\mathrm{train}}$ give the best result at every $\epsilon_{\mathrm{test}}$?
2. What happens to clean accuracy as the training budget changes?
3. Does robustness degrade smoothly as $\epsilon_{\mathrm{test}}$ increases?
4. Which training budget gives the best result under your validation criterion?
5. What conclusion would be unjustified if you evaluated only at $\epsilon_{\mathrm{test}}=\epsilon_{\mathrm{train}}$?


**Write your response here:**

<br><br><br><br>


## 6. Compare untargeted and targeted adversarial training

The previous sections used untargeted FGSM to study the main design variables. The defense can also generate targeted adversarial examples during training.

In targeted adversarial training, the target class is used to **construct the adversarial input**. When the defender updates the model, that adversarial input is still trained with its **ground-truth label**.

The target-selection policy is therefore part of the defense design.


### Question 9 — How should target classes be selected?

Consider three possible policies:

1. one fixed target class;
2. a random incorrect target for each attacked example;
3. a target selected from the current model outputs.

For each policy:

1. Which source-target relationships can it expose the model to?
2. What bias can the policy introduce?
3. How much additional computation does it require?
4. Could a narrow target policy make the defense appear stronger than it is?
5. Which policy will you use for training, and why?
6. Should one target or several targets be generated for each clean example? Discuss the trade-off.


**Write your response here:**

<br><br><br><br><br>


### Implementation 6 — Train untargeted and targeted defenses

Using the design choices selected in Sections 2–5, train comparable untargeted and targeted adversarially trained models.


In [29]:
# 1. Implement the target-selection policy chosen in Question 9.
#
# 2. Extend the adversarial-example generation utility so that it supports both
#    untargeted and targeted FGSM training.
#
# 3. Train an untargeted adversarially trained model.
#
# 4. Train a targeted adversarially trained model using the same architecture,
#    data protocol, optimizer family, number of epochs, rho, lambda, and epsilon_train.
#
# 5. Keep the clean baseline for comparison.
#
# 6. Record clean validation accuracy, training time, and the settings of all three models.


### Question 10 — What should transfer between the two defenses?

Before evaluating the models, state your predictions.

1. Do you expect untargeted adversarial training to improve resistance to targeted attacks?
2. Do you expect targeted adversarial training to improve resistance to untargeted attacks?
3. Must transfer occur in both directions?
4. What result would suggest that a defense has specialized to the objective used during training?
5. What targeted evaluations would be needed before making a broad claim about targeted robustness?


**Write your response here:**

<br><br><br><br><br>


## 7. Evaluate the defended models

A white-box evaluation must attack the model that is actually being evaluated. The final comparison should therefore use fresh adversarial examples generated separately for each model.

Evaluate the clean baseline, the untargeted adversarially trained model, and the targeted adversarially trained model under both attack objectives.


### Question 11 — What makes the evaluation fair?

1. Why is reusing adversarial examples generated for a different model not the same as a direct white-box evaluation?
2. Which quantities must be recomputed for each model?
3. Which attack settings must be identical when comparing models?
4. For targeted evaluation, how will you avoid drawing conclusions from only one convenient source-target pair?
5. Why should clean accuracy be shown beside the adversarial metrics?
6. What result would indicate a robustness gain that is specific to one attack objective?


**Write your response here:**

<br><br><br><br><br>


### Implementation 7 — Final validation comparison

Evaluate each model with fresh untargeted and targeted white-box attacks.


In [30]:
# 1. Implement fresh untargeted white-box evaluation for an arbitrary model.
#
# 2. Implement fresh targeted white-box evaluation using a declared target protocol.
#
# 3. Use the same evaluation budgets and eligibility rules for all models.
#
# 4. Evaluate the clean baseline, untargeted-trained model, and targeted-trained model.
#
# 5. Report clean accuracy, robust accuracy, untargeted ASR, and targeted ASR.
#
# 6. For targeted evaluation, use several source-target pairs or a target-sampling
#    protocol that is applied consistently across models.
#
# 7. Present one compact comparison table and the plots needed to support your conclusions.


### Question 12 — What did the final comparison show?

1. Which model is strongest against untargeted FGSM?
2. Which model is strongest against targeted FGSM under your target protocol?
3. Is robustness transfer between targeted and untargeted training symmetric?
4. What evidence suggests specialization, if any?
5. Which result surprised you most, and what follow-up experiment would you run to investigate it?


**Write your response here:**

<br><br><br><br><br>


## 8. Measure the computational cost of the defense

Adversarial training adds attack-generation work to ordinary training. Robustness should therefore be considered together with computation.


### Question 13 — Where is the extra computation?

Inspect your implementation.

1. Which operations are needed only to generate adversarial inputs?
2. Which operations are needed to update the model parameters?
3. Which gradients are actually required during attack generation?
4. Which evaluation operations can run without gradients?
5. During an FGSM sweep over several values of $\epsilon_{\mathrm{test}}$, which quantities can be reused?
6. Which timing measurements are useful when comparing defense configurations?


**Write your response here:**

<br><br><br><br>


### Implementation 8 — Improve one part of the implementation

Profile one representative experiment and remove at least one unnecessary computation without changing the mathematical procedure.


In [31]:
# 1. Measure the runtime of one representative training or evaluation configuration.
#
# 2. Identify one redundant computation or unnecessary gradient calculation.
#
# 3. Refactor the implementation while preserving the same mathematical procedure.
#
# 4. Re-run the same configuration.
#
# 5. Compare runtime and numerical results before and after the change.
#
# 6. Explain why the refactoring is mathematically equivalent.


## 9. Select the final defense

At this point, $\rho$, $\lambda$, $\epsilon_{\mathrm{train}}$, generation strategy, and attack objective have all been studied. The final configuration must now be selected using validation evidence only.


### Question 14 — Which configuration should be selected?

1. Define the rule you will use to choose the final defended model.
2. How will clean accuracy enter the rule?
3. How will untargeted and targeted robustness enter the rule?
4. How will computational cost enter the rule?
5. If no model satisfies the minimum acceptable clean performance, what should you report?
6. Why must the test set remain unused until this choice has been fixed?


**Write your response here:**

<br><br><br><br><br>


### Implementation 9 — Held-out test evaluation

Fix the final defense using the validation rule, then perform the held-out evaluation.


In [32]:
# 1. Apply the validation rule from Question 14 and select one final defense configuration.
#
# 2. Fix all hyperparameters and attack-evaluation settings before examining the test results.
#
# 3. If your protocol requires retraining after model selection, retrain without using the test set for tuning.
#
# 4. Evaluate the clean baseline and selected defense once on the held-out test set.
#
# 5. Report clean accuracy, untargeted robust accuracy and ASR, targeted ASR under
#    the declared target protocol, and the relevant training/evaluation cost.
#
# 6. Store the final results in one table.


## 10. State the robustness claim

This laboratory provides empirical evidence under a specific threat model, attack algorithm, perturbation constraint, and set of experimental choices. The conclusion must not be broader than the evidence.


### Question 15 — What can you claim?

Write two statements:

1. the strongest robustness claim supported by your results;
2. a stronger claim that your results do **not** support.

Your supported claim must state:

- white-box attacker knowledge;
- the $\ell_\infty$ perturbation constraint;
- the FGSM attack algorithm;
- the evaluated perturbation budgets;
- whether the evidence concerns untargeted attacks, targeted attacks, or both.

Finally, identify one stronger evaluation that should be performed next and explain what additional evidence it would provide.


**Write your response here:**

<br><br><br><br><br><br>


## 11. Submission

Submit the completed notebook and a short technical summary.

The summary should contain:

1. the experimental protocol;
2. the static-versus-online comparison;
3. the effect of $\rho$;
4. the effect of $\lambda$;
5. the $\epsilon_{\mathrm{train}}$ versus $\epsilon_{\mathrm{test}}$ analysis;
6. the targeted-versus-untargeted training comparison;
7. the selected final defense;
8. the held-out test results;
9. the supported robustness claim and its limitations.

Use tables and figures only when they support a specific conclusion. Every reported metric must state the condition under which it was measured.

### References

[1] A. Vassilev, A. Oprea, A. Fordyce, H. Anderson, X. Davies, and M. Hamin, *Adversarial Machine Learning: A Taxonomy and Terminology of Attacks and Mitigations*, NIST AI 100-2e2025, National Institute of Standards and Technology, 2025.

[2] I. J. Goodfellow, J. Shlens, and C. Szegedy, “Explaining and Harnessing Adversarial Examples,” *International Conference on Learning Representations*, 2015.
